# Public vs Private Collections — customer demo

A two-scenario walkthrough that mirrors the real customer use cases:

- **Scenario A — Public collection**: writes go through the catalog API, items land in PostgreSQL, the OUTBOX drains them into the shared `dynastore-items` Elasticsearch alias. Anyone (including anonymous callers) can search them via STAC.
- **Scenario B — Private collection**: same write path, but the collection is marked `is_private=true`. Items are still indexed for authenticated search, but anonymous STAC search must return zero results and never leak content.

Both scenarios run end-to-end against the live review environment.

`DYNASTORE_TOKEN` is required for write operations. The script reads it from the environment (or `.env` via `python-dotenv` when available).

In [ ]:
import json
import os
import time
import uuid

import httpx

try:
    from dotenv import load_dotenv  # type: ignore
    load_dotenv()
except Exception:
    pass

BASE_URL = os.environ.get(
    "DYNASTORE_BASE_URL",
    "https://data.review.fao.org/geospatial/v2/api/catalog",
)
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:6])

PUB_CATALOG_ID  = f"demo-public-{RUN_ID}"
PRIV_CATALOG_ID = f"demo-private-{RUN_ID}"
PUB_COLL_ID     = f"public-coll-{RUN_ID}"
PRIV_COLL_ID    = f"private-coll-{RUN_ID}"

auth_headers: dict = {}
if TOKEN:
    auth_headers["Authorization"] = f"Bearer {TOKEN}"

# Authenticated client for writes & authenticated probes.
client = httpx.Client(base_url=BASE_URL, headers=auth_headers, timeout=60.0)
# Anonymous client for the privacy probes — no Authorization header.
anon = httpx.Client(base_url=BASE_URL, timeout=60.0)

print(f"Base URL : {BASE_URL}")
print(f"Run ID   : {RUN_ID}")
print(f"Auth     : {'token set' if TOKEN else 'ANONYMOUS — write ops will 401'}")

## Helpers — readiness polls

Two small helpers used throughout the demo:

- `wait_for_catalog(cat_id)` — tenant provisioning is async, so a request issued in the same millisecond as the catalog POST can still 404. Poll `GET /stac/catalogs/{cat}` until it answers (200/401/403 all prove the tenant exists).
- `wait_for_search(cat_id)` — items go through PostgreSQL → OUTBOX → Elasticsearch. Search returns nothing until the drain completes; poll the search endpoint instead of guessing a fixed sleep.

In [ ]:
def wait_for_catalog(cat_id: str, timeout_s: int = 30, poll_s: float = 1.0) -> None:
    deadline = time.monotonic() + timeout_s
    last = None
    while time.monotonic() < deadline:
        r = client.get(f"/stac/catalogs/{cat_id}")
        last = r.status_code
        if r.status_code in (200, 401, 403):
            return
        time.sleep(poll_s)
    raise RuntimeError(
        f"Catalog {cat_id!r} not reachable after {timeout_s}s (last={last})"
    )


def wait_for_search(cat_id: str, expected: int, timeout_s: int = 60, poll_s: float = 5.0) -> int:
    deadline = time.monotonic() + timeout_s
    last_count = 0
    while time.monotonic() < deadline:
        r = client.post(
            f"/search/catalogs/{cat_id}/items-search",
            content=json.dumps({"limit": expected * 2 or 10}),
            headers={"Content-Type": "application/json"},
        )
        if r.is_success:
            last_count = len(r.json().get("features", []))
            if last_count >= expected:
                return last_count
        time.sleep(poll_s)
    return last_count

---
## Scenario A — Public collection

Items land in the shared ES alias and are searchable by anonymous callers.

### A.1 — Create the catalog and wait for it to be reachable

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PUB_CATALOG_ID,
    "type": "Catalog", "stac_version": "1.0.0",
    "title": f"Demo public {RUN_ID}",
    "description": "Customer demo — safe to delete",
    "links": [],
}))
if r.status_code not in (200, 201, 409):
    raise RuntimeError(f"catalog create failed: {r.status_code} {r.text[:300]}")
print(f"Catalog {PUB_CATALOG_ID!r}: {r.status_code}")
wait_for_catalog(PUB_CATALOG_ID)
print(f"Catalog {PUB_CATALOG_ID!r} ready")

### A.2 — Create the public collection

In [ ]:
r = client.post(f"/stac/catalogs/{PUB_CATALOG_ID}/collections", content=json.dumps({
    "id": PUB_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Public items demo", "links": [], "title": "Public Items",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "license": "proprietary",
}))
if r.status_code not in (200, 201, 409):
    raise RuntimeError(f"collection create failed: {r.status_code} {r.text[:300]}")
print(f"Collection {PUB_COLL_ID!r}: {r.status_code}")

### A.3 — Ingest 3 public STAC items

In [ ]:
PUB_ITEM_IDS = []
for i in range(3):
    item_id = f"pub-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PUB_CATALOG_ID}/collections/{PUB_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0",
            "id": item_id,
            "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.1, 41.9]},
            "bbox": [12.4 + i * 0.1, 41.8, 12.6 + i * 0.1, 42.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z"},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PUB_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code} ok")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:200]}")

if len(PUB_ITEM_IDS) != 3:
    raise RuntimeError(f"only {len(PUB_ITEM_IDS)}/3 items ingested — see errors above")

### A.4 — Authenticated STAC search returns the items

Polls until the OUTBOX has drained the new items into Elasticsearch.

In [ ]:
count = wait_for_search(PUB_CATALOG_ID, expected=3, timeout_s=60)
print(f"authenticated search: {count} feature(s) after polling")
if count >= 3:
    print("  OK — ES routing + OUTBOX drain are working")
else:
    # Distinguish OUTBOX-not-drained from a real routing problem by
    # forcing the PG path with an explicit hint.
    r = client.post(
        f"/search/catalogs/{PUB_CATALOG_ID}/items-search?hint=geometry_exact",
        content=json.dumps({"limit": 10}),
    )
    pg = len(r.json().get("features", [])) if r.is_success else 0
    if pg >= 3:
        raise RuntimeError(
            f"items present in PG ({pg}) but invisible via the ES alias — "
            "check that this catalog is registered in the dynastore-items alias."
        )
    raise RuntimeError(
        f"items missing from both ES ({count}) and PG ({pg}) — ingest failure."
    )

### A.5 — Anonymous STAC search also returns the items

Same query, no `Authorization` header. A public collection must be visible to everyone.

In [ ]:
r = anon.post(
    f"/search/catalogs/{PUB_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
    headers={"Content-Type": "application/json"},
)
print(f"anonymous search: HTTP {r.status_code}")
feats = r.json().get("features", []) if r.is_success else []
print(f"  features: {len(feats)}")
if len(feats) >= 3:
    print("  OK — public items visible without auth")
else:
    raise RuntimeError("public collection should be readable anonymously but returned 0 features")

---
## Scenario B — Private collection

Same code path; flipping `is_private=true` on the collection denormalises the privacy bit into Elasticsearch so anonymous callers see nothing.

### B.1 — Create the private catalog + collection

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PRIV_CATALOG_ID,
    "type": "Catalog", "stac_version": "1.0.0",
    "title": f"Demo private {RUN_ID}",
    "description": "Customer demo — safe to delete",
    "links": [],
}))
if r.status_code not in (200, 201, 409):
    raise RuntimeError(f"catalog create failed: {r.status_code} {r.text[:300]}")
print(f"Catalog {PRIV_CATALOG_ID!r}: {r.status_code}")
wait_for_catalog(PRIV_CATALOG_ID)

r = client.post(f"/stac/catalogs/{PRIV_CATALOG_ID}/collections", content=json.dumps({
    "id": PRIV_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Private items demo", "links": [], "title": "Private Items",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "license": "proprietary",
}))
if r.status_code not in (200, 201, 409):
    raise RuntimeError(f"collection create failed: {r.status_code} {r.text[:300]}")
print(f"Collection {PRIV_COLL_ID!r}: {r.status_code}")

### B.2 — Mark the collection private

`PUT /configs/catalogs/{cat}/collections/{coll}/plugins/collection_privacy` flips the bit; the apply-handler propagates `is_private=true` into the collection's ES doc.

In [ ]:
r = client.put(
    f"/configs/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/plugins/collection_privacy",
    content=json.dumps({"is_private": True}),
)
if not r.is_success:
    raise RuntimeError(f"privacy config write failed: {r.status_code} {r.text[:300]}")
print(f"PUT collection_privacy: {r.status_code}")
print(f"  value: {r.json()}")

### B.3 — Ingest 3 private items

In [ ]:
PRIV_ITEM_IDS = []
for i in range(3):
    item_id = f"priv-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0",
            "id": item_id,
            "geometry": {"type": "Point", "coordinates": [13.5 + i * 0.1, 42.9]},
            "bbox": [13.4 + i * 0.1, 42.8, 13.6 + i * 0.1, 43.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z", "confidential": True},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PRIV_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code} ok")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:200]}")

if len(PRIV_ITEM_IDS) != 3:
    raise RuntimeError(f"only {len(PRIV_ITEM_IDS)}/3 private items ingested")

### B.4 — Authenticated user sees the private items

In [ ]:
count = wait_for_search(PRIV_CATALOG_ID, expected=3, timeout_s=60)
print(f"authenticated search: {count} feature(s)")
if count < 3:
    raise RuntimeError("authenticated user should see all private items")

### B.5 — Anonymous caller sees nothing

The same query without an `Authorization` header must return either 401/403 or 0 features. A non-empty result here would be a privacy leak and the cell will raise.

In [ ]:
r = anon.post(
    f"/search/catalogs/{PRIV_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
    headers={"Content-Type": "application/json"},
)
print(f"anonymous search: HTTP {r.status_code}")
if r.status_code in (401, 403):
    print("  OK — anonymous access rejected at the auth layer")
elif r.is_success and not r.json().get("features", []):
    print("  OK — anonymous search returned 0 features (privacy enforced)")
else:
    feats = r.json().get("features", []) if r.is_success else []
    raise RuntimeError(
        f"PRIVACY LEAK: anonymous caller got {len(feats)} private feature(s); "
        f"first id: {feats[0].get('id') if feats else None}"
    )

---
## Recap

| Scenario | Authenticated | Anonymous |
|---|---|---|
| Public collection — STAC search | OK (A.4) | OK (A.5) |
| Private collection — STAC search | OK (B.4) | DENY / 0 features (B.5) |

The same write path serves both — the only difference is the `collection_privacy` config bit set in B.2.

## Teardown

In [ ]:
for cat_id in [PUB_CATALOG_ID, PRIV_CATALOG_ID]:
    r = client.delete(f"/stac/catalogs/{cat_id}", params={"force": "true"})
    print(f"DELETE {cat_id!r}: {r.status_code}")
client.close()
anon.close()
print("Done.")